# 🕉️ SankhyaVox — Complete Pipeline
**Synthetic Data → Feature Extraction → Multi-Model Training → Evaluation**

Run each cell in order. This notebook uses **4 TTS engines** and **class-balanced training** to build robust models for 0–99 recognition.

---
## 📦 Step 0: Install Dependencies
Run this cell ONCE. Restart kernel after if needed.

In [1]:
!pip install gTTS edge-tts pyttsx3 imageio-ffmpeg librosa soundfile hmmlearn scikit-learn scipy joblib

# Try Coqui TTS (optional - needs PyTorch)
try:
    import torch
    !pip install TTS
    print("✅ Coqui TTS installed (PyTorch found)")
except ImportError:
    print("⚠️ PyTorch not found — Coqui TTS skipped (gTTS + edge-tts + pyttsx3 will still generate 1800+ samples)")

✅ Coqui TTS installed (PyTorch found)


ERROR: Ignored the following versions that require a different python version: 0.0.10.2 Requires-Python >=3.6.0, <3.9; 0.0.10.3 Requires-Python >=3.6.0, <3.9; 0.0.11 Requires-Python >=3.6.0, <3.9; 0.0.12 Requires-Python >=3.6.0, <3.9; 0.0.13.1 Requires-Python >=3.6.0, <3.9; 0.0.13.2 Requires-Python >=3.6.0, <3.9; 0.0.14.1 Requires-Python >=3.6.0, <3.9; 0.0.15 Requires-Python >=3.6.0, <3.9; 0.0.15.1 Requires-Python >=3.6.0, <3.9; 0.0.9 Requires-Python >=3.6.0, <3.9; 0.0.9.1 Requires-Python >=3.6.0, <3.9; 0.0.9.2 Requires-Python >=3.6.0, <3.9; 0.0.9a10 Requires-Python >=3.6.0, <3.9; 0.0.9a9 Requires-Python >=3.6.0, <3.9; 0.1.0 Requires-Python >=3.6.0, <3.10; 0.1.1 Requires-Python >=3.6.0, <3.10; 0.1.2 Requires-Python >=3.6.0, <3.10; 0.1.3 Requires-Python >=3.6.0, <3.10; 0.10.0 Requires-Python >=3.7.0, <3.11; 0.10.1 Requires-Python >=3.7.0, <3.11; 0.10.2 Requires-Python >=3.7.0, <3.11; 0.11.0 Requires-Python >=3.7.0, <3.11; 0.11.1 Requires-Python >=3.7.0, <3.11; 0.12.0 Requires-Python >=3

---
## 🔧 Step 1: Setup & Imports

In [3]:
import os, sys, glob, tempfile, subprocess, warnings
import numpy as np
import soundfile as sf
import librosa
import joblib
warnings.filterwarnings('ignore')

PROJECT_ROOT = r"C:\Users\rakes\OneDrive\Desktop\Sujal\SankhyaVox"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

from src.config import (
    VOCAB, SAMPLE_RATE, FEATURE_DIR, MODEL_DIR,
    HMM_STATES, BAUM_WELCH_ITERS, TOKEN_TO_IDX, IDX_TO_TOKEN
)
from scripts.extract_features import preprocess_audio, extract_mfcc, process_file

import imageio_ffmpeg
ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()

RAW_SYNTH_DIR = os.path.join("data", "synthetic_raw")
os.makedirs(RAW_SYNTH_DIR, exist_ok=True)
os.makedirs(str(FEATURE_DIR), exist_ok=True)

print(f"✅ Project: {PROJECT_ROOT}")
print(f"✅ Vocab: {len(VOCAB)} tokens → {VOCAB}")
print(f"✅ FFmpeg: {ffmpeg_exe[:50]}...")

✅ Project: C:\Users\rakes\OneDrive\Desktop\Sujal\SankhyaVox
✅ Vocab: 13 tokens → ['shunya', 'eka', 'dvi', 'tri', 'catur', 'pancha', 'shat', 'sapta', 'ashta', 'nava', 'dasha', 'vimsati', 'shata']
✅ FFmpeg: C:\Users\rakes\anaconda3\Lib\site-packages\imageio...


---
## 🗺️ Step 2: Text Maps & Augmentation Engine

In [5]:
# Sequence augmentation removed. Real data only.
print('Skipping augmentation...')


✅ Augmentation engine: 30 variants per voice
✅ 13 single tokens + 27 multi-word sequences


---
## 🔊 Step 3: Generate — Google TTS (5 accents)

In [7]:
from gtts import gTTS

GTTS_TLDS = ['co.in', 'com', 'co.uk', 'com.au', 'ca']

print("=== gTTS: Single Tokens ===")
for token in VOCAB:
    if token not in DEVANAGARI_MAP: continue
    for tld in GTTS_TLDS:
        print(f"  {token} ({tld})", end=".. ", flush=True)
        with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as f: tmp = f.name
        try:
            gTTS(text=DEVANAGARI_MAP[token], lang='hi', tld=tld).save(tmp)
            wav = tmp.replace('.mp3','.wav')
            subprocess.run([ffmpeg_exe,'-y','-i',tmp,'-ar',str(SAMPLE_RATE),'-ac','1',wav],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
            a, sr = sf.read(wav); augment_and_save(a, sr, token, f"gtts_{tld.replace('.','')}"); os.remove(wav)
            print("✓", end=" ")
        except Exception as e: print(f"✗", end=" ")
        finally:
            if os.path.exists(tmp): os.remove(tmp)
    print()
print(f"\n📊 Running total: {generated_count} features")

=== gTTS: Single Tokens ===
  shunya (co.in).. ✓   shunya (com).. ✓   shunya (co.uk).. ✓   shunya (com.au).. ✓   shunya (ca).. ✓ 
  eka (co.in).. ✓   eka (com).. ✓   eka (co.uk).. ✓   eka (com.au).. ✓   eka (ca).. ✓ 
  dvi (co.in).. ✓   dvi (com).. ✓   dvi (co.uk).. ✓   dvi (com.au).. ✓   dvi (ca).. ✓ 
  tri (co.in).. ✓   tri (com).. ✓   tri (co.uk).. ✓   tri (com.au).. ✓   tri (ca).. ✓ 
  catur (co.in).. ✓   catur (com).. ✓   catur (co.uk).. ✓   catur (com.au).. ✓   catur (ca).. ✓ 
  pancha (co.in).. ✓   pancha (com).. ✓   pancha (co.uk).. ✓   pancha (com.au).. ✓   pancha (ca).. ✓ 
  shat (co.in).. ✓   shat (com).. ✓   shat (co.uk).. ✓   shat (com.au).. ✓   shat (ca).. ✓ 
  sapta (co.in).. ✓   sapta (com).. ✓   sapta (co.uk).. ✓   sapta (com.au).. ✓   sapta (ca).. ✓ 
  ashta (co.in).. ✓   ashta (com).. ✓   ashta (co.uk).. ✓   ashta (com.au).. ✓   ashta (ca).. ✓ 
  nava (co.in).. ✓   nava (com).. ✓   nava (co.uk).. ✓   nava (com.au).. ✓   nava (ca).. ✓ 
  dasha (co.in).. ✓   dasha (com

---
## 🔊 Step 4: Generate — Microsoft Edge Neural TTS (2 voices)

In [9]:
EDGE_VOICES = ['hi-IN-SwaraNeural', 'hi-IN-MadhurNeural']

print("=== Edge-TTS: Single Tokens ===")
for token in VOCAB:
    if token not in DEVANAGARI_MAP: continue
    for voice in EDGE_VOICES:
        print(f"  {token} ({voice.split('-')[-1]})", end=".. ", flush=True)
        with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as f: tmp = f.name
        try:
            subprocess.run(['edge-tts','--voice',voice,'--text',DEVANAGARI_MAP[token],'--write-media',tmp],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
            wav = tmp.replace('.mp3','.wav')
            subprocess.run([ffmpeg_exe,'-y','-i',tmp,'-ar',str(SAMPLE_RATE),'-ac','1',wav],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
            a, sr = sf.read(wav); augment_and_save(a, sr, token, voice.replace('-','')); os.remove(wav)
            print("✓", end=" ")
        except: print("✗", end=" ")
        finally:
            if os.path.exists(tmp): os.remove(tmp)
    print()
print(f"\n📊 Running total: {generated_count} features")

=== Edge-TTS: Single Tokens ===
  shunya (SwaraNeural).. ✓   shunya (MadhurNeural).. ✓ 
  eka (SwaraNeural).. ✓   eka (MadhurNeural).. ✓ 
  dvi (SwaraNeural).. ✓   dvi (MadhurNeural).. ✓ 
  tri (SwaraNeural).. ✓   tri (MadhurNeural).. ✓ 
  catur (SwaraNeural).. ✓   catur (MadhurNeural).. ✓ 
  pancha (SwaraNeural).. ✓   pancha (MadhurNeural).. ✓ 
  shat (SwaraNeural).. ✓   shat (MadhurNeural).. ✓ 
  sapta (SwaraNeural).. ✓   sapta (MadhurNeural).. ✓ 
  ashta (SwaraNeural).. ✓   ashta (MadhurNeural).. ✓ 
  nava (SwaraNeural).. ✓   nava (MadhurNeural).. ✓ 
  dasha (SwaraNeural).. ✓   dasha (MadhurNeural).. ✓ 
  vimsati (SwaraNeural).. ✓   vimsati (MadhurNeural).. ✓ 
  shata (SwaraNeural).. ✓   shata (MadhurNeural).. ✓ 

📊 Running total: 2730 features


---
## 🔊 Step 5: Generate — pyttsx3 Offline (Windows SAPI5)

In [22]:
import itertools
from joblib import Parallel, delayed

# 1. 🚀 FAST PARALLEL AUGMENTATION ENGINE
def _process_single_aug(args):
    a_copy, sr, p, s, n, out_path, feat_dir = args
    try:
        if s != 1.0: a_copy = librosa.effects.time_stretch(y=a_copy, rate=s)
        if p != 0: a_copy = librosa.effects.pitch_shift(y=a_copy, sr=sr, n_steps=p)
        if n > 0: a_copy = a_copy + np.random.normal(0, n, len(a_copy)).astype(np.float32)
        sf.write(out_path, a_copy.astype(np.float32), sr)
        process_file(out_path, feat_dir)
        return 1
    except: return 0

def augment_and_save(audio, sr, token, var_name):
    global generated_count
    trimmed, _ = librosa.effects.trim(audio, top_db=30)
    if len(trimmed) > int(sr * 0.1): audio = trimmed
    
    tasks = []
    # Generate all 30 variations
    for p, s, n in itertools.product(PITCH_STEPS, SPEED_RATES, NOISE_LEVELS):
        out = os.path.join(RAW_SYNTH_DIR, f"synth_{var_name}_p{p}_s{int(s*100)}_n{int(n*10000)}_{token}.wav")
        feat_dir = os.path.join(str(FEATURE_DIR), f"synth_{var_name}")
        tasks.append((audio.copy(), sr, p, s, n, out, feat_dir))
        
    # Run augmentations in parallel using all CPU cores!
    results = Parallel(n_jobs=-1, prefer="threads")(delayed(_process_single_aug)(t) for t in tasks)
    generated_count += sum(results)


# 2. ⚡ FAST PYTTSX3 CELL (With Fast Resampling)
print("=== pyttsx3: Offline SAPI5 ===")
try:
    import pyttsx3
    engine = pyttsx3.init()
    engine.setProperty('rate', 150)
    for token in VOCAB:
        if token not in DEVANAGARI_MAP: continue
        print(f"  {token}", end=".. ", flush=True)
        tmp = os.path.join(tempfile.gettempdir(), f"pyttsx3_{token}.wav")
        engine.save_to_file(DEVANAGARI_MAP[token], tmp)
        engine.runAndWait()
        
        if os.path.exists(tmp) and os.path.getsize(tmp) > 1000:
            a, sr = sf.read(tmp)
            if sr != SAMPLE_RATE: 
                # Use 'kaiser_fast' to drastically speed up resampling
                a = librosa.resample(a, orig_sr=sr, target_sr=SAMPLE_RATE, res_type='kaiser_fast')
            augment_and_save(a, SAMPLE_RATE, token, 'pyttsx3')
            print("✓", end=" ")
        else: print("✗", end=" ")
        
        if os.path.exists(tmp): os.remove(tmp)
    engine.stop()
    print()
except Exception as e: print(f"⚠️ pyttsx3 skipped: {e}")
print(f"\n📊 Running total: {generated_count} features")


=== pyttsx3: Offline SAPI5 ===
  shunya.. 

KeyboardInterrupt: 

---
## 🔊 Step 6 (Optional): Generate — Coqui TTS
This only runs if you have PyTorch + Coqui TTS installed.

In [18]:
print("=== Coqui TTS (Optional) ===")
try:
    from TTS.api import TTS
    # Use Coqui's multilingual XTTS or a Hindi model
    tts_model = TTS(model_name="tts_models/multilingual/multi-dataset/xtts_v2", progress_bar=False)
    print("✅ Coqui TTS loaded!")
    
    for token in VOCAB:
        if token not in DEVANAGARI_MAP: continue
        print(f"  {token}", end=".. ", flush=True)
        tmp = os.path.join(tempfile.gettempdir(), f"coqui_{token}.wav")
        try:
            tts_model.tts_to_file(text=DEVANAGARI_MAP[token], file_path=tmp, language="hi")
            if os.path.exists(tmp) and os.path.getsize(tmp) > 1000:
                a, sr = sf.read(tmp)
                if sr != SAMPLE_RATE: a = librosa.resample(a, orig_sr=sr, target_sr=SAMPLE_RATE)
                augment_and_save(a, SAMPLE_RATE, token, 'coqui')
                print("✓", end=" ")
            else: print("✗", end=" ")
        except: print("✗", end=" ")
        if os.path.exists(tmp): os.remove(tmp)
    print()
except ImportError:
    print("⚠️ Coqui TTS not installed — skipping (not required, other engines are sufficient)")
except Exception as e:
    print(f"⚠️ Coqui TTS error: {e}")
print(f"\n📊 Running total: {generated_count} features")

=== Coqui TTS (Optional) ===
⚠️ Coqui TTS not installed — skipping (not required, other engines are sufficient)

📊 Running total: 2730 features


---
## 🔗 Step 7: Generate Multi-Word Sequences (CRITICAL for double digits!)
This synthesizes full phrases like "दश पञ्च" (15) and splits them into word-level segments.
Without this, the HMM cannot accurately decode two-digit numbers.

In [20]:
print("Sequence generation removed. Grammar-constrained decoder handles co-articulation at inference time.")


=== Multi-Word Sequences ===
  dasha_eka: g✓ g✓ e✓ e✓ 
  dasha_dvi: g✓ g✓ e✓ e✓ 
  dasha_tri: g✓ g✓ e✓ e✓ 
  dasha_catur: g✓ g✓ e✓ e✓ 
  dasha_pancha: g✓ g✓ e✓ e✓ 
  dasha_shat: g✓ g✓ e✓ e✓ 
  dasha_sapta: g✓ g✓ e✓ e✓ 
  dasha_ashta: g✓ g✓ e✓ e✓ 
  dasha_nava: g✓ g✓ e✓ e✓ 
  vimsati_eka: g✓ g✓ e✓ e✓ 
  vimsati_pancha: g✓ g✓ e✓ e✓ 
  vimsati_sapta: g✓ g✓ e✓ e✓ 
  vimsati_nava: g✓ g✓ e✓ e✓ 
  tri_dasha: g✓ g✓ e✓ e✓ 
  tri_dasha_pancha: g✓ g✓ e✓ e✓ 
  catur_dasha: g✓ g✓ e✓ e✓ 
  catur_dasha_dvi: g✓ g✓ e✓ e✓ 
  pancha_dasha: g✓ g✓ e✓ e✓ 
  pancha_dasha_sapta: g✓ g✓ e✓ e✓ 
  shat_dasha: g✓ g✓ e✓ e✓ 
  shat_dasha_tri: g✓ g✓ e✓ e✓ 
  sapta_dasha: g✓ g✓ e✓ e✓ 
  sapta_dasha_dvi: g✓ g✓ e✓ e✓ 
  ashta_dasha: g✓ g✓ e✓ e✓ 
  ashta_dasha_pancha: g✓ g✓ e✓ e✓ 
  nava_dasha: g✓ g✓ e✓ e✓ 
  nava_dasha_nava: g✓ g✓ e✓ e✓ 

📊 TOTAL features generated: 10050


---
## 📊 Step 8: Data Diagnostics

In [24]:
feature_files = glob.glob(os.path.join(str(FEATURE_DIR), "**", "*.npy"), recursive=True)
print(f"Total .npy files: {len(feature_files)}")

counts = {v: 0 for v in VOCAB}
for fp in feature_files:
    bn = os.path.basename(fp)
    for v in VOCAB:
        if f"_{v}_" in bn or bn.startswith(v+"_") or f"_{v}." in bn:
            counts[v] += 1; break

print(f"\n{'Token':<12} {'Count':>6}  Status")
print("-" * 32)
for tok, cnt in counts.items():
    s = "✅" if cnt >= 50 else ("⚠️" if cnt >= 20 else "❌")
    print(f"{tok:<12} {cnt:>6}  {s}")
print(f"{'TOTAL':<12} {sum(counts.values()):>6}")

Total .npy files: 6505

Token         Count  Status
--------------------------------
shunya          405  ✅
eka             525  ✅
dvi             525  ✅
tri             525  ✅
catur           525  ✅
pancha          525  ✅
shat            525  ✅
sapta           525  ✅
ashta           525  ✅
nava            525  ✅
dasha           525  ✅
vimsati         485  ✅
shata           365  ✅
TOTAL          6505


---
## 🧠 Step 9: Train All 3 Models (Class-Balanced!)

In [26]:
from scripts.train_v2 import train_hmm_models_v2
import os

MODEL_DIR = 'models/'
os.makedirs(MODEL_DIR, exist_ok=True)
print('Training per-token HMMs...')
hmm = train_hmm_models_v2('data/features/', VOCAB, HMM_STATES, MODEL_DIR)


Loaded 6505 feature files.

  HMM TRAINING (3 GMM Mixtures)
Training HMM for 'shunya' with 6 states, 3 mixtures, 405 examples...


  File "C:\Users\rakes\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\rakes\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\rakes\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\rakes\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


Training HMM for 'eka' with 5 states, 3 mixtures, 525 examples...
Training HMM for 'dvi' with 5 states, 3 mixtures, 525 examples...
Training HMM for 'tri' with 5 states, 3 mixtures, 525 examples...
Training HMM for 'catur' with 7 states, 3 mixtures, 525 examples...
Training HMM for 'pancha' with 7 states, 3 mixtures, 525 examples...
Training HMM for 'shat' with 5 states, 3 mixtures, 525 examples...
Training HMM for 'sapta' with 7 states, 3 mixtures, 525 examples...
Training HMM for 'ashta' with 7 states, 3 mixtures, 525 examples...
Training HMM for 'nava' with 6 states, 3 mixtures, 525 examples...
Training HMM for 'dasha' with 6 states, 3 mixtures, 525 examples...
Training HMM for 'vimsati' with 8 states, 3 mixtures, 485 examples...
Training HMM for 'shata' with 5 states, 3 mixtures, 365 examples...
Models saved to C:\Users\rakes\OneDrive\Desktop\Sujal\SankhyaVox\models\sankhya_hmm_models.pkl
✅ HMM saved.

Fixed dataset: (6505, 195), 13 classes

  SVM TRAINING (class_weight=balanced)
✅

---
## 🧪 Step 10: Evaluation

In [28]:
from src.decoder import GrammarConstrainedDecoder
from src.grammar import all_valid_sequences, _TOKEN_TO_VALUE

hmm2 = SankhyaHMMBase(VOCAB, HMM_STATES, str(MODEL_DIR))
hmm2.load_models()
decoder = GrammarConstrainedDecoder(hmm2, all_valid_sequences)

tests = glob.glob(os.path.join(str(FEATURE_DIR), "**", "*.npy"), recursive=True)
np.random.seed(42); np.random.shuffle(tests)

print(f"{'File':<40} {'True':>5} {'Pred':>5} Result")
print("-" * 60)
correct, total = 0, 0
for fp in tests[:30]:
    bn = os.path.basename(fp)
    true_tok = None
    for v in VOCAB:
        if f"_{v}_" in bn or f"_{v}." in bn: true_tok = v; break
    if not true_tok: continue
    pred, score = decoder.decode(np.load(fp))
    true_n = _TOKEN_TO_VALUE.get(true_tok, -1)
    total += 1
    ok = pred == true_n
    if ok: correct += 1
    print(f"{bn[:38]:<40} {true_n:>5} {pred:>5} {'✅' if ok else '❌'}")
print("-" * 60)
if total: print(f"Accuracy: {correct}/{total} = {correct/total*100:.1f}%")

Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance


File                                      True  Pred Result
------------------------------------------------------------


Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance


synth_hiINMadhurNeural_p0_s114_n40_eka       1     1 ✅


Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance


...

synth_hiINMadhurNeural_p-2_s100_n40_na       9     9 ✅


Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance


synth_gtts_com_p0_s114_n0_tri.npy            3     3 ✅


Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance
Degenerate mixture covariance


synth_gtts_ca_p-2_s85_n0_shunya.npy          0     0 ✅
------------------------------------------------------------
Accuracy: 26/30 = 86.7%


---
## 🚀 Step 11: Launch Web App
Run `python app.py` in your terminal → open http://127.0.0.1:5000

In [30]:
print("Run in terminal: python app.py")
print("Open: http://127.0.0.1:5000")

Run in terminal: python app.py
Open: http://127.0.0.1:5000
